# Preparing Text for LLMs

Download, clean, and tokenize *The Time Machine* by H.G. Wells (Project Gutenberg). Goal: a word-level vocabulary with integer encode/decode.

In [10]:
import numpy as np
import requests
import re

## 1. Download

In [11]:
response = requests.get('https://www.gutenberg.org/files/35/35-0.txt')
text = response.content.decode('utf-8')

print(f'Downloaded {len(text):,} characters')
text[:1000]

Downloaded 182,973 characters


'*** START OF THE PROJECT GUTENBERG EBOOK 35 ***\r\n\r\n\r\n\r\n\r\nThe Time Machine\r\n\r\nAn Invention\r\n\r\nby H. G. Wells\r\n\r\n\r\nCONTENTS\r\n\r\n I Introduction\r\n II The Machine\r\n III The Time Traveller Returns\r\n IV Time Travelling\r\n V In the Golden Age\r\n VI The Sunset of Mankind\r\n VII A Sudden Shock\r\n VIII Explanation\r\n IX The Morlocks\r\n X When Night Came\r\n XI The Palace of Green Porcelain\r\n XII In the Darkness\r\n XIII The Trap of the White Sphinx\r\n XIV The Further Vision\r\n XV The Time Traveller’s Return\r\n XVI After the Story\r\n Epilogue\r\n\r\n\r\n\r\n\r\n I.\r\n Introduction\r\n\r\n\r\nThe Time Traveller (for so it will be convenient to speak of him) was\r\nexpounding a recondite matter to us. His pale grey eyes shone and\r\ntwinkled, and his usually pale face was flushed and animated. The fire\r\nburnt brightly, and the soft radiance of the incandescent lights in the\r\nlilies of silver caught the bubbles that flashed and passed in our\r\nglas

## 2. Clean

In [12]:
# Trim Gutenberg header/footer
start = text.find('*** START OF THE PROJECT GUTENBERG')
end   = text.find('*** END OF THE PROJECT GUTENBERG')
text  = text[text.index('\n', start):end]

# Normalize: line endings and emphasis markers, then drop non-ASCII
# (this cleanly handles curly quotes, em-dashes, etc. without hardcoding them)
text = text.replace('\r\n', '\n').replace('_', ' ')
text = re.sub(r'[^\x00-\x7F]+', ' ', text)
text = re.sub(r'\d+', '', text)
text = re.sub(r'\s+', ' ', text).lower().strip()

text[:500]

'the time machine an invention by h. g. wells contents i introduction ii the machine iii the time traveller returns iv time travelling v in the golden age vi the sunset of mankind vii a sudden shock viii explanation ix the morlocks x when night came xi the palace of green porcelain xii in the darkness xiii the trap of the white sphinx xiv the further vision xv the time traveller s return xvi after the story epilogue i. introduction the time traveller (for so it will be convenient to speak of him)'

## 3. Tokenize

In [13]:
# Split on anything non-alphabetic; text is already lowercased so [^a-z] catches all separators
words = re.split(r'[^a-z]+', text)
words = [w for w in words if len(w) > 1]

nWords = len(words)
print(f'{nWords:,} total words')
words[:20]

30,686 total words


['the',
 'time',
 'machine',
 'an',
 'invention',
 'by',
 'wells',
 'contents',
 'introduction',
 'ii',
 'the',
 'machine',
 'iii',
 'the',
 'time',
 'traveller',
 'returns',
 'iv',
 'time',
 'travelling']

## 4. Vocabulary

In [14]:
vocab = sorted(set(words))
nLex  = len(vocab)

print(f'{nWords:,} words, {nLex:,} unique tokens')

30,686 words, 4,586 unique tokens


## 5. Encode & Decode

In [15]:
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = dict(enumerate(vocab))

list(word2idx.items())[:10]

[('abandon', 0),
 ('abandoned', 1),
 ('able', 2),
 ('abnormally', 3),
 ('abominable', 4),
 ('abominations', 5),
 ('about', 6),
 ('above', 7),
 ('abruptly', 8),
 ('absence', 9)]

In [16]:
def encode(words, w2i=word2idx):
    return np.array([w2i[w] for w in words], dtype=int)

def decode(indices, i2w=idx2word):
    return ' '.join(i2w[i] for i in indices)

In [17]:
print(encode(['the', 'time', 'machine']))
print(decode([1, 3, 10]))

[4039 4106 2414]
abandoned abnormally absent


In [18]:
start  = np.random.randint(nWords - 10)
sample = words[start:start + 10]

encoded = encode(sample)
decoded = decode(encoded)

print('Original:', sample)
print('Indices: ', encoded)
print('Decoded: ', decoded)
assert sample == decoded.split(), 'Roundtrip failed!'

Original: ['of', 'the', 'nights', 'of', 'our', 'acquaintance', 'including', 'the', 'last', 'night']
Indices:  [2729 4039 2659 2729 2778   35 2054 4039 2273 2657]
Decoded:  of the nights of our acquaintance including the last night
